# 01. Regresión Lineal: Los Fundamentos del Machine Learning

**Nivel:** 🟢 Principiante  
**Tiempo estimado:** 60 minutos  
**Prerequisitos:** Ninguno (este es el punto de partida)

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, podrás:
- Entender qué es la regresión lineal y cuándo usarla
- Derivar matemáticamente la función de costo (MSE)
- Implementar regresión lineal desde cero usando solo NumPy
- Comparar tu implementación con Scikit-learn
- Interpretar los parámetros aprendidos (weights y bias)
- Evaluar el rendimiento usando métricas apropiadas

In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Importar utilidades personalizadas
import sys
sys.path.append('../../shared/utils')
from visualization import plot_regression_line, plot_loss_history
from testing import test_exercise, check_shape, check_close
from datasets import load_dataset, generate_synthetic_regression

# Semilla para reproducibilidad
np.random.seed(42)

print("✅ Librerías importadas correctamente")

---
## 📌 1. Motivación: ¿Por qué Regresión Lineal?

### El Problema del Mundo Real

Imagina que eres un analista de datos en una inmobiliaria. Tu jefe te pide:

> *"Necesito predecir el precio de una casa basándome en su tamaño. Tenemos datos históricos de 100 ventas. ¿Puedes crear un modelo?"*

Este es un problema de **regresión**: queremos predecir un valor continuo (precio) basándonos en una o más características (tamaño).

### ¿Por qué es importante?

La regresión lineal es:
- 📊 **El modelo más simple** de Machine Learning supervisado
- 🧠 **La base conceptual** para entender modelos más complejos (redes neuronales son generalizaciones)
- 🔧 **Increíblemente útil** en la práctica para miles de aplicaciones
- 📈 **Interpretable**: puedes explicar exactamente cómo funciona

### Aplicaciones Reales

- 🏠 Predicción de precios de viviendas
- 📈 Pronóstico de ventas
- 🌡️ Modelado de relaciones causa-efecto en ciencias
- 💰 Estimación de salarios basados en experiencia
- 📊 Análisis de tendencias en datos financieros

### La Pregunta Guía

Al final de este notebook responderemos:

> **¿Cómo encuentra un algoritmo la "mejor" línea que pasa por un conjunto de puntos, y qué significa "mejor"?**

---
## 📊 2. Intuición Visual: Entendiendo la Regresión Lineal

Antes de entrar en la matemática, construyamos intuición visual.

In [ ]:
# Generar datos sintéticos simples para visualización
np.random.seed(42)
X_simple = np.linspace(0, 10, 50)
y_simple = 2.5 * X_simple + 1.0 + np.random.normal(0, 2, 50)

# Crear figura interactiva
fig = go.Figure()

# Puntos de datos
fig.add_trace(go.Scatter(
    x=X_simple,
    y=y_simple,
    mode='markers',
    name='Datos Reales',
    marker=dict(size=10, color='blue', opacity=0.6),
    hovertemplate='Tamaño: %{x:.1f}<br>Precio: %{y:.1f}<extra></extra>'
))

# Línea de regresión "verdadera" (la que generó los datos)
y_true_line = 2.5 * X_simple + 1.0
fig.add_trace(go.Scatter(
    x=X_simple,
    y=y_true_line,
    mode='lines',
    name='Línea Verdadera (y = 2.5x + 1.0)',
    line=dict(color='green', width=2, dash='dash')
))

fig.update_layout(
    title="Ejemplo de Regresión Lineal: Tamaño vs Precio de Casas",
    xaxis_title="Tamaño de la casa (100s de pies cuadrados)",
    yaxis_title="Precio ($100,000s)",
    template="plotly_white",
    font=dict(size=12),
    hovermode='closest'
)

fig.show()

print("\n💡 Observa:")
print("   • Los puntos azules son los datos reales (con ruido natural)")
print("   • La línea verde es la relación 'verdadera' que generó los datos")
print("   • Nuestro objetivo: encontrar una línea que se ajuste lo mejor posible a los puntos")

### Visualización Interactiva: ¿Qué pasa con diferentes líneas?

Veamos cómo diferentes valores de pendiente (m) y bias (b) afectan el ajuste.

In [ ]:
# Función para calcular el error (MSE)
def calculate_mse(X, y, m, b):
    y_pred = m * X + b
    mse = np.mean((y - y_pred) ** 2)
    return mse

# Probar diferentes parámetros
test_params = [
    (1.0, 5.0, 'Mala (m=1.0, b=5.0)'),
    (2.0, 2.0, 'Regular (m=2.0, b=2.0)'),
    (2.5, 1.0, 'Óptima (m=2.5, b=1.0)')
]

fig = go.Figure()

# Datos reales
fig.add_trace(go.Scatter(
    x=X_simple,
    y=y_simple,
    mode='markers',
    name='Datos',
    marker=dict(size=8, color='blue', opacity=0.6)
))

# Diferentes líneas
colors = ['red', 'orange', 'green']
for (m, b, label), color in zip(test_params, colors):
    y_line = m * X_simple + b
    mse = calculate_mse(X_simple, y_simple, m, b)
    
    fig.add_trace(go.Scatter(
        x=X_simple,
        y=y_line,
        mode='lines',
        name=f'{label} - MSE={mse:.2f}',
        line=dict(color=color, width=2)
    ))

fig.update_layout(
    title="Comparación de Diferentes Líneas de Regresión",
    xaxis_title="X",
    yaxis_title="y",
    template="plotly_white",
    font=dict(size=12)
)

fig.show()

print("\n💡 Observaciones Clave:")
print("   • El MSE (Mean Squared Error) mide qué tan 'lejos' está la línea de los datos")
print("   • Un MSE menor indica un mejor ajuste")
print("   • La línea verde (óptima) tiene el MSE más bajo")
print("   • Nuestro objetivo: encontrar automáticamente los valores óptimos de m y b")

---
## 🧮 3. Fundamentos Matemáticos

Ahora que tenemos la intuición, formalicemos los conceptos.

### 📖 Notación

| Símbolo | Significado |
|---------|-------------|
| $m$ | Número de ejemplos de entrenamiento |
| $n$ | Número de features (características) |
| $x^{(i)}$ | Features del ejemplo $i$ |
| $y^{(i)}$ | Target (valor real) del ejemplo $i$ |
| $\hat{y}^{(i)}$ | Predicción para el ejemplo $i$ |
| $w$ | Pesos (weights) del modelo |
| $b$ | Bias (intercepto) |

### El Modelo de Regresión Lineal

La forma más simple (una feature):

$$
\begin{align}
\hat{y} &= wx + b \tag{1}
\end{align}
$$

Forma general (múltiples features):

$$
\begin{align}
\hat{y} &= w_1 x_1 + w_2 x_2 + ... + w_n x_n + b \tag{2} \\
\hat{y} &= \mathbf{w}^T \mathbf{x} + b \tag{3}
\end{align}
$$

Donde:
- $\mathbf{w} = [w_1, w_2, ..., w_n]^T$ es el vector de pesos
- $\mathbf{x} = [x_1, x_2, ..., x_n]^T$ es el vector de features
- $b$ es el bias (término independiente)

### La Función de Costo (MSE)

Necesitamos una forma de medir qué tan "bueno" es nuestro modelo. Usamos el **Mean Squared Error (MSE)**:

$$
\begin{align}
J(w, b) &= \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})^2 \tag{4} \\
&= \frac{1}{m} \sum_{i=1}^{m} (w^T x^{(i)} + b - y^{(i)})^2 \tag{5}
\end{align}
$$

**¿Por qué elevamos al cuadrado?**
1. Para hacer todos los errores positivos (no se cancelan)
2. Penaliza más los errores grandes
3. Es matemáticamente conveniente (diferenciable)

### Ejemplo Numérico Concreto

Supongamos que tenemos 3 casas:

| Casa | Tamaño (x) | Precio Real (y) | Predicción ($\hat{y}$) | Error | Error² |
|------|-----------|-----------------|----------------------|-------|--------|
| 1 | 1000 | 150 | 160 | -10 | 100 |
| 2 | 1500 | 200 | 195 | 5 | 25 |
| 3 | 2000 | 250 | 230 | 20 | 400 |

$$
MSE = \frac{100 + 25 + 400}{3} = \frac{525}{3} = 175
$$

In [ ]:
# Verificar el ejemplo numérico
sizes = np.array([1000, 1500, 2000])
prices = np.array([150, 200, 250])
predictions = np.array([160, 195, 230])

errors = predictions - prices
squared_errors = errors ** 2
mse = np.mean(squared_errors)

print("Verificación del Ejemplo Numérico:")
print(f"Errores: {errors}")
print(f"Errores al cuadrado: {squared_errors}")
print(f"MSE: {mse}")
print(f"✅ Coincide con nuestro cálculo manual: {mse == 175}")

### Encontrando los Parámetros Óptimos

**Objetivo:** Encontrar $w$ y $b$ que minimicen $J(w, b)$

Hay dos enfoques principales:

#### 1. Solución Analítica (Normal Equation)

Para regresión lineal, existe una fórmula cerrada:

$$
w = (X^T X)^{-1} X^T y \tag{6}
$$

**Ventajas:** Una sola operación, resultado exacto

**Desventajas:** Lento para muchas features ($O(n^3)$), requiere invertir matriz

#### 2. Gradiente Descendente (Iterativo)

Actualizamos los parámetros iterativamente:

$$
\begin{align}
w &:= w - \alpha \frac{\partial J}{\partial w} \tag{7} \\
b &:= b - \alpha \frac{\partial J}{\partial b} \tag{8}
\end{align}
$$

Donde $\alpha$ es el **learning rate**.

Los gradientes son:

$$
\begin{align}
\frac{\partial J}{\partial w} &= \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)}) \cdot x^{(i)} \tag{9} \\
\frac{\partial J}{\partial b} &= \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)}) \tag{10}
\end{align}
$$

En el siguiente notebook exploraremos Gradient Descent en profundidad. Por ahora, usaremos la solución analítica.

---
## 💻 4. Implementación Desde Cero

Ahora implementemos regresión lineal usando **solo NumPy**.

In [ ]:
class RegresionLineal:
    """
    Implementación desde cero de Regresión Lineal.
    
    Esta clase implementa regresión lineal usando dos métodos:
    1. Normal Equation (solución analítica)
    2. Gradient Descent (solución iterativa)
    
    Parameters:
    -----------
    learning_rate : float, default=0.01
        Tasa de aprendizaje para gradient descent
    n_iterations : int, default=1000
        Número de iteraciones para gradient descent
    method : str, default='gradient_descent'
        Método a usar: 'gradient_descent' o 'normal_equation'
    """
    
    def __init__(self, learning_rate=0.01, n_iterations=1000, method='gradient_descent'):
        self.lr = learning_rate
        self.n_iter = n_iterations
        self.method = method
        
        # Parámetros del modelo (se aprenden durante fit)
        self.weights = None
        self.bias = None
        
        # Para tracking
        self.losses = []  # Historial de pérdidas
    
    def fit(self, X, y):
        """
        Entrena el modelo de regresión lineal.
        
        Parameters:
        -----------
        X : np.ndarray, shape (n_samples, n_features)
            Features de entrenamiento
        y : np.ndarray, shape (n_samples,)
            Target values
        
        Returns:
        --------
        self : RegresionLineal
            Modelo entrenado
        """
        # Asegurar que X es 2D
        if len(X.shape) == 1:
            X = X.reshape(-1, 1)
        
        n_samples, n_features = X.shape
        
        if self.method == 'normal_equation':
            # Solución analítica usando la Normal Equation
            # Agregamos columna de 1s para el bias
            X_b = np.c_[np.ones((n_samples, 1)), X]
            
            # Fórmula: theta = (X^T * X)^(-1) * X^T * y
            theta = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
            
            # Separar bias y weights
            self.bias = theta[0]
            self.weights = theta[1:]
            
            # Calcular loss final
            y_pred = self.predict(X)
            loss = np.mean((y - y_pred) ** 2)
            self.losses = [loss]  # Solo un valor
            
            print(f"✅ Entrenamiento completado (Normal Equation)")
            print(f"   Loss final: {loss:.4f}")
        
        else:  # gradient_descent
            # Inicialización de parámetros
            self.weights = np.zeros(n_features)
            self.bias = 0
            
            # Gradiente descendente
            for i in range(self.n_iter):
                # 1. Forward pass: calcular predicciones
                y_pred = X @ self.weights + self.bias
                
                # 2. Calcular loss (MSE)
                loss = np.mean((y - y_pred) ** 2)
                self.losses.append(loss)
                
                # 3. Calcular gradientes
                # dL/dw = -2/n * sum((y - y_pred) * x)
                # dL/db = -2/n * sum(y - y_pred)
                error = y_pred - y
                dw = (1 / n_samples) * (X.T @ error)
                db = (1 / n_samples) * np.sum(error)
                
                # 4. Actualizar parámetros
                self.weights -= self.lr * dw
                self.bias -= self.lr * db
                
                # Logging cada 100 iteraciones
                if i % 100 == 0:
                    print(f"Iteración {i:4d} - Loss: {loss:.4f}")
            
            print(f"\n✅ Entrenamiento completado")
            print(f"   Loss final: {self.losses[-1]:.4f}")
            print(f"   Weights: {self.weights}")
            print(f"   Bias: {self.bias:.4f}")
        
        return self
    
    def predict(self, X):
        """
        Hace predicciones usando el modelo entrenado.
        
        Parameters:
        -----------
        X : np.ndarray, shape (n_samples, n_features)
            Features para predicción
        
        Returns:
        --------
        y_pred : np.ndarray, shape (n_samples,)
            Predicciones
        """
        if self.weights is None:
            raise ValueError("El modelo no ha sido entrenado. Llama a fit() primero.")
        
        # Asegurar que X es 2D
        if len(X.shape) == 1:
            X = X.reshape(-1, 1)
        
        # Predicción: y = X * w + b
        return X @ self.weights + self.bias
    
    def score(self, X, y):
        """
        Calcula el coeficiente de determinación R² del modelo.
        
        R² = 1 - (SS_res / SS_tot)
        Donde:
        - SS_res = sum((y - y_pred)^2)  # Suma de residuos cuadrados
        - SS_tot = sum((y - y_mean)^2)  # Suma total de cuadrados
        
        R² = 1 significa ajuste perfecto
        R² = 0 significa que el modelo no es mejor que predecir la media
        
        Parameters:
        -----------
        X : np.ndarray
            Features
        y : np.ndarray
            Target values
        
        Returns:
        --------
        r2 : float
            Coeficiente R²
        """
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        r2 = 1 - (ss_res / ss_tot)
        return r2

print("✅ Clase RegresionLineal definida")

### Probemos nuestra implementación con datos sintéticos

In [ ]:
# Generar datos de prueba
X_train, X_test, y_train, y_test = generate_synthetic_regression(
    n_samples=100,
    n_features=1,
    noise=10.0,
    random_state=42
)

print(f"📊 Datos generados:")
print(f"   Entrenamiento: {X_train.shape[0]} ejemplos")
print(f"   Prueba: {X_test.shape[0]} ejemplos")
print(f"   Features: {X_train.shape[1]}")

In [ ]:
# Entrenar nuestro modelo con Gradient Descent
print("🔧 Entrenando con Gradient Descent...\n")
modelo_gd = RegresionLineal(learning_rate=0.01, n_iterations=1000, method='gradient_descent')
modelo_gd.fit(X_train, y_train)

In [ ]:
# Visualizar la evolución del loss
fig = plot_loss_history(modelo_gd.losses, title="Evolución del Loss durante el Entrenamiento")
fig.show()

print("\n💡 Observa cómo el loss disminuye rápidamente al inicio y luego se estabiliza.")
print("   Esto indica que el algoritmo está convergiendo al mínimo.")

In [ ]:
# Evaluar el modelo
y_pred_train = modelo_gd.predict(X_train)
y_pred_test = modelo_gd.predict(X_test)

mse_train = mean_squared_error(y_train, y_pred_train)
mse_test = mean_squared_error(y_test, y_pred_test)
r2_train = modelo_gd.score(X_train, y_train)
r2_test = modelo_gd.score(X_test, y_test)

print("📊 Resultados de Evaluación:")
print(f"\n   Entrenamiento:")
print(f"   - MSE: {mse_train:.4f}")
print(f"   - R²: {r2_train:.4f}")
print(f"\n   Prueba:")
print(f"   - MSE: {mse_test:.4f}")
print(f"   - R²: {r2_test:.4f}")

print("\n💡 Interpretación del R²:")
print(f"   El modelo explica el {r2_test*100:.1f}% de la varianza en los datos de prueba.")

In [ ]:
# Visualizar el ajuste
fig = plot_regression_line(
    X_test,
    y_test,
    modelo_gd.weights,
    modelo_gd.bias,
    title="Regresión Lineal: Datos de Prueba vs Predicciones"
)
fig.show()

### Comparación: Gradient Descent vs Normal Equation

In [ ]:
# Entrenar con Normal Equation
print("🔧 Entrenando con Normal Equation...\n")
modelo_ne = RegresionLineal(method='normal_equation')
modelo_ne.fit(X_train, y_train)

# Comparar parámetros
print("\n📊 Comparación de Parámetros:")
print(f"\n   Gradient Descent:")
print(f"   - Weight: {modelo_gd.weights[0]:.4f}")
print(f"   - Bias: {modelo_gd.bias:.4f}")
print(f"\n   Normal Equation:")
print(f"   - Weight: {modelo_ne.weights[0]:.4f}")
print(f"   - Bias: {modelo_ne.bias:.4f}")

# Comparar R² en test
r2_gd = modelo_gd.score(X_test, y_test)
r2_ne = modelo_ne.score(X_test, y_test)

print(f"\n   R² en Test:")
print(f"   - Gradient Descent: {r2_gd:.4f}")
print(f"   - Normal Equation: {r2_ne:.4f}")

print("\n💡 Ambos métodos deberían dar resultados muy similares.")
print("   La Normal Equation es exacta pero más lenta para muchas features.")
print("   Gradient Descent es aproximado pero escala mejor.")

---
## 🏭 5. Versión con Framework (Scikit-learn)

Ahora comparemos nuestra implementación con la de Scikit-learn.

In [ ]:
# Entrenar modelo de Scikit-learn
sklearn_model = LinearRegression()
sklearn_model.fit(X_train, y_train)

# Predicciones
y_pred_sklearn = sklearn_model.predict(X_test)

# Métricas
mse_sklearn = mean_squared_error(y_test, y_pred_sklearn)
r2_sklearn = r2_score(y_test, y_pred_sklearn)

print("📊 Comparación: Nuestra Implementación vs Scikit-learn")
print("\n" + "="*60)
print(f"{'Métrica':<20} {'Nuestra':<15} {'Scikit-learn':<15}")
print("="*60)
print(f"{'Weight':<20} {modelo_ne.weights[0]:<15.4f} {sklearn_model.coef_[0]:<15.4f}")
print(f"{'Bias':<20} {modelo_ne.bias:<15.4f} {sklearn_model.intercept_:<15.4f}")
print(f"{'MSE (Test)':<20} {mse_test:<15.4f} {mse_sklearn:<15.4f}")
print(f"{'R² (Test)':<20} {r2_test:<15.4f} {r2_sklearn:<15.4f}")
print("="*60)

print("\n✅ ¡Nuestros resultados coinciden con Scikit-learn!")
print("   Esto valida que nuestra implementación es correcta.")

### Ventajas de usar Scikit-learn

- ✅ **Optimizado**: Implementaciones en C/Cython muy rápidas
- ✅ **Robusto**: Maneja casos edge automáticamente
- ✅ **API consistente**: Mismo patrón fit/predict para todos los modelos
- ✅ **Features adicionales**: Cross-validation, regularización, etc.
- ✅ **Bien documentado**: Excelente documentación y ejemplos

### Cuándo usar cada enfoque

**Implementación propia:**
- 🎓 Para aprender y entender los algoritmos
- 🔧 Para modificar/experimentar con variantes
- 📝 Para propósitos educativos

**Scikit-learn:**
- 🚀 Para proyectos en producción
- ⏱️ Cuando el tiempo es crítico
- 📊 Para análisis estándar

---
## 🎯 6. Ejercicios Prácticos

Ahora es tu turno de aplicar lo aprendido.

### 🟢 Ejercicio 1: Predicción Simple

Carga el dataset de California Housing y entrena un modelo de regresión lineal para predecir precios de casas basándote en el ingreso medio (`MedInc`).

In [ ]:
def ejercicio_1():
    """
    Objetivo: Entrenar un modelo de regresión lineal con datos reales
    
    Instrucciones:
    1. Carga el dataset 'california_housing' usando load_dataset()
    2. Usa SOLO la primera feature (MedInc)
    3. Entrena un modelo RegresionLineal con method='normal_equation'
    4. Retorna el R² en el conjunto de prueba
    
    Returns:
    --------
    r2 : float
        Coeficiente R² en test set
    """
    # TODO: Tu código aquí
    # X_train, X_test, y_train, y_test = load_dataset('california_housing', ...)
    # X_train = X_train[:, 0:1]  # Solo primera feature
    # X_test = X_test[:, 0:1]
    # modelo = ...
    # ...
    # return r2
    
    pass

# Descomentar para probar
# resultado = ejercicio_1()
# print(f"\n📊 R² obtenido: {resultado:.4f}")
# print("💡 Un R² > 0.40 es un buen resultado para una sola feature")

### 🟡 Ejercicio 2: Regresión Multivariable

Ahora usa TODAS las features del dataset California Housing para mejorar la predicción.

In [ ]:
def ejercicio_2():
    """
    Objetivo: Comparar regresión con 1 feature vs todas las features
    
    Instrucciones:
    1. Carga el dataset completo (todas las features)
    2. Entrena dos modelos:
       - Modelo A: Solo primera feature
       - Modelo B: Todas las features
    3. Compara sus R² en test
    
    Returns:
    --------
    tuple : (r2_una_feature, r2_todas_features)
    """
    # TODO: Tu código aquí
    
    pass

# Descomentar para probar
# r2_1, r2_all = ejercicio_2()
# print(f"\n📊 Resultados:")
# print(f"   1 feature: R² = {r2_1:.4f}")
# print(f"   Todas: R² = {r2_all:.4f}")
# print(f"   Mejora: {(r2_all - r2_1)/r2_1*100:.1f}%")

### 🔴 Ejercicio 3: Análisis de Residuos

Los residuos son las diferencias entre predicciones y valores reales. Un buen modelo debe tener residuos que:
1. Estén distribuidos normalmente alrededor de 0
2. No muestren patrones (indica que el modelo lineal es apropiado)

In [ ]:
def ejercicio_3():
    """
    Objetivo: Analizar los residuos de tu modelo
    
    Instrucciones:
    1. Entrena un modelo con el dataset California Housing (todas features)
    2. Calcula los residuos: residuos = y_test - y_pred
    3. Crea dos visualizaciones:
       a) Histograma de residuos (debería verse como una campana centrada en 0)
       b) Scatter plot de residuos vs predicciones (no debería haber patrón)
    4. Calcula la media y desviación estándar de los residuos
    
    Returns:
    --------
    dict : {'mean': float, 'std': float}
    """
    # TODO: Tu código aquí
    # Pista: usa plotly o matplotlib para las visualizaciones
    # fig = go.Figure()
    # fig.add_trace(go.Histogram(x=residuos, ...)
    
    pass

# Descomentar para probar
# stats = ejercicio_3()
# print(f"\n📊 Estadísticas de Residuos:")
# print(f"   Media: {stats['mean']:.4f} (debería estar cerca de 0)")
# print(f"   Std: {stats['std']:.4f}")

---
## 📚 7. Resumen y Recursos

### 🎯 Puntos Clave

1. **Regresión Lineal** es el modelo más fundamental de ML, usado para predecir valores continuos

2. **El modelo** se representa como $\hat{y} = w^T x + b$, donde buscamos los mejores $w$ y $b$

3. **La función de costo** MSE mide qué tan bueno es el modelo: $J(w,b) = \frac{1}{m} \sum (\hat{y}^{(i)} - y^{(i)})^2$

4. **Dos métodos de optimización**:
   - Normal Equation: Solución exacta pero costosa computacionalmente
   - Gradient Descent: Aproximación iterativa que escala mejor

5. **Métricas importantes**:
   - MSE: Error promedio al cuadrado (menor es mejor)
   - R²: Porcentaje de varianza explicada (1 es perfecto, 0 es malo)

6. **Implementar desde cero** te da comprensión profunda, pero **usa frameworks** en producción

### 🔗 Recursos Adicionales

#### 📄 Papers Fundamentales

- **"Least Squares"** - Gauss & Legendre (1800s)
  - Historia: Este método tiene más de 200 años y sigue siendo relevante
  - El método de mínimos cuadrados fue desarrollado independientemente por Gauss y Legendre

#### 📖 Libros Recomendados

- **"Introduction to Statistical Learning"** - James, Witten, Hastie, Tibshirani
  - Capítulo 3: Linear Regression (muy accesible)
  - Disponible gratis online

- **"Pattern Recognition and Machine Learning"** - Bishop
  - Capítulo 3.1: Linear Basis Function Models
  - Más matemático pero muy completo

#### 🎥 Videos Recomendados

- **Andrew Ng - Machine Learning Course (Coursera)**
  - Semanas 1-2 cubren regresión lineal en detalle
  - Excelente para visualizar los conceptos

- **3Blue1Brown - "Essence of Linear Algebra"**
  - Ayuda a entender las operaciones matriciales
  - Fundamenta la intuición geométrica

#### 💻 Documentación

- [Scikit-learn: Linear Regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- [NumPy Documentation](https://numpy.org/doc/stable/)

### 🤔 Preguntas para Reflexionar

1. ¿Qué pasa si tus features tienen escalas muy diferentes (ej: tamaño en miles, edad en decenas)?
   - Pista: Normalización

2. ¿Cómo sabes si una relación lineal es apropiada para tus datos?
   - Pista: Análisis de residuos

3. ¿Qué hacer cuando el modelo funciona bien en train pero mal en test?
   - Pista: Overfitting, exploraremos en notebooks siguientes

---

## ➡️ Próximo Paso

En el siguiente notebook, **02. Gradient Descent**, profundizaremos en:

- Cómo funciona realmente el gradiente descendente
- Diferentes variantes (Batch, Stochastic, Mini-batch)
- El rol crítico del learning rate
- Técnicas de optimización (momentum, Adam)
- Visualización de la superficie de pérdida

El gradient descent no solo es fundamental para regresión lineal, sino que es **el algoritmo de optimización base de todas las redes neuronales**.

---

<div align="center">

**🎉 ¡Felicidades por completar tu primer notebook! 🎉**

**Continúa con: [02. Gradient Descent](02-gradient-descent.ipynb)**

[← Volver al índice de ML Clásico](README.md)

</div>